In [1]:
import pandas as pd
import numpy as np
#Mettre des noms sur les colonnes
colonnes = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
            "Insulin", "IMC", "DiabetesPedigreeFunction", "Age", "Outcome"]

df = pd.read_csv("pima-indians-diabetes.csv", names=colonnes)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,IMC,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Objectif 
- 1° Y a-t-il des valeurs manquantes ou aberrantes ?
- 2° Qui sont les patientes de ce dataset ? (âge moyen, distribution)
- 3° Quelle est la prévalence du diabète dans cet échantillon ? Est-ce que cette proportion ressemble à la réalité, ou est-ce un échantillon biaisé ?
- 4° Y a-t-il des différences entre les groupes diabétiques et non-diabétiques ? (comparer les moyennes)
- 5° Quelles variables semblent le plus associées au diabète ? Est-ce que ça correspond à ce que je sais déjà de la physiopathologie du diabète ?

In [2]:
#Remplacer les valeurs manquantes 
colonnes_a_nettoyer = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "IMC", "DiabetesPedigreeFunction", "Age"]
df[colonnes_a_nettoyer] = df[colonnes_a_nettoyer].replace(0, np.nan)

#Combien y a-t-il de valeurs manquantes ?
print(df.isnull().sum())

#Est-ce qu'il y a des valeurs aberrantes (maximum)
df["Insulin"].max()


Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
IMC                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


np.float64(846.0)

On peut voir qu'il y a beaucoup de valeurs manquantes dans les colonnes insuline et épaisseur de la peau. On en a 374 sur 768 pour l'insuline (49%), 227 sur 768 pour l'épaisseur de peau (soit 30 %). Les valeurs aberrantes comme les maximums doivent être prises en considération, par exemple une insuline à 846.

In [3]:
#Déterminer la représentativité de l'échantillon en calculant le pourcentage de personnes dans chaque tranche de l'échelle d'IMC
bins = [0, 18.5, 25, 30, 35, 40, float("inf")]
labels = ["Maigreur", "Normal", "Surpoids", "Obésité I", "Obésité II", "Obésité III"]
classes = pd.cut(df["IMC"], bins=bins, labels=labels, right=False)
print(round((classes.value_counts(normalize=True).sort_index() * 100), 2))

#Value_counts(normalize=True) -> compte les lignes par tranche et renvoie des proportions (entre 0 et 1) au lieu de nombres bruts.
#.sort_index() remet les tranches dans l'ordre naturel (sinon elles sont triées de la plus fréquente à la moins fréquente).

IMC
Maigreur        0.53
Normal         13.47
Surpoids       23.65
Obésité I      29.59
Obésité II     19.82
Obésité III    12.95
Name: proportion, dtype: float64


Autre façon de la faire = utiliser une fonction, if row.IMC < 18.5 et elif pour toutes les autres tranches. Puis faire 'df.apply()' sur axis columns. Attention, il faut mettre 'pd.isnull()' dans le premier if, pour retirer toutes les valeurs manquantes (NaN). Les résultats obtenus avant pd.isnull() pour la catégorie obésité III sont de 14,58 %. 

Voici les statistiques dans la population française d'après l'Enquête épidémiologique nationale (ObÉpi-Roche / OFEO) mise à jour par la Ligue contre l'obésité :
- 1- Insuffisance pondérale (Maigreur) | IMC inférieur à 18,5 | 4,5 %
- 2- Corpulence normale | IMC entre 18,5 et 24,9 | 48,2 %
- 3- Surpoids | IMC entre 25,0 et 29,9 | 30,3 %
- 4- Obésité modérée (Classe I) | IMC entre 30,0 et 34,9 | 11,9 %
- 5- Obésité sévère (Classe II) | IMC entre 35 et 39,9 | 3,1 %
- 6- Obésité morbide (Classe III) | IMC > 40 | 2 %

Nous avons environ 62,4 % des patientes en obésité (classe I, II et III), contre 17 % dans la population française. Ceci constitue un biais.

Attention, l'enquête ObEpi est auto-déclarée, les IMC sont sûrement sous-estimés.

In [4]:
#Déterminer l'âge moyen
moyenne_age = df["Age"].mean()
print("La moyenne d'âge est de", round(moyenne_age, 1))

#Déterminer la médiane de l'âge
médiane_age = df.Age.median()
print("La médiane d'âge est de", médiane_age)

#Distribution de l'âge 
print("Voici la distribution complète","\n",df["Age"].describe())

#Répartition par tranche d'âge
bins = [20, 30, 40, 50, float("inf")]
labels = ["Entre 20 et 29", "Entre 30 et 39", "Entre 40 et 49", "50 et au-dessus"]
repartition_age = pd.cut(df.Age, bins=bins, labels=labels, right=False)
print(repartition_age.value_counts(normalize=True).sort_index())

#Le taux de diabète change-t-il d'une tranche à l'autre
taux_diabetes_par_tranche = (df.groupby(repartition_age)["Outcome"].mean()*100)
print("Taux de diabète par tranche d'âge :", taux_diabetes_par_tranche)



La moyenne d'âge est de 33.2
La médiane d'âge est de 29.0
Voici la distribution complète 
 count    768.000000
mean      33.240885
std       11.760232
min       21.000000
25%       24.000000
50%       29.000000
75%       41.000000
max       81.000000
Name: Age, dtype: float64
Age
Entre 20 et 29     0.515625
Entre 30 et 39     0.214844
Entre 40 et 49     0.153646
50 et au-dessus    0.115885
Name: proportion, dtype: float64
Taux de diabète par tranche d'âge : Age
Entre 20 et 29     21.212121
Entre 30 et 39     46.060606
Entre 40 et 49     55.084746
50 et au-dessus    48.314607
Name: Outcome, dtype: float64


/var/folders/ms/t1f4y3395_x8ltyn1d1hrrzh0000gn/T/ipykernel_44169/2137209378.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  taux_diabetes_par_tranche = (df.groupby(repartition_age)["Outcome"].mean()*100)


La médiane (29 ans) est inférieure à la moyenne (33 ans) : la distribution est asymétrique, avec une majorité de jeunes patientes et une minorité plus âgée qui tire la moyenne vers le haut. La médiane décrit donc mieux la patiente typique de cet échantillon. Ces données ne peuvent pas être généralisées à une population plus âgée. De plus, on peut voir que le taux de diabète dans la tranche de 20 à 29 ans, là où nous avons 51,6 % des femmes de l'échantillon, n'est que de 21,2 %, c'est-à-dire le plus faible des 4 tranches.

In [5]:
#Déterminer le pourcentage de personnes avec une PAD >= 90
personne_avec_hta = df["BloodPressure"].map(lambda p : p >= 90).sum()
pourcentage_personne_hta = personne_avec_hta / df["BloodPressure"].count()
print(f"Pourcentage de personnes avec PAD supérieur ou égale à 90 mmHg = {pourcentage_personne_hta:.1%}")

Pourcentage de personnes avec PAD supérieur ou égale à 90 mmHg = 8.2%


D'après Santé publique France, la prévalence globale mesurée de l'hypertension artérielle est de 30,6% dans la population adulte. En regardant selon l'âge, elle est de 6,3 % pour les 18-34 ans et de 67,8 % pour les 65-74 ans. Donc notre taux (8,2 %) de personnes avec une PAD supérieur ou égale à 90, qui peut s'apparenter à de l'HTA, est supérieur à celui de la population des 18-34 ans. On le compare à ce dernier puisque 75 % de l'échantillon ont 41 ans ou moins. 

Cette comparaison reste approximative. En effet, Santé Publique France définit l'HTA par une PAS ≥ 140 et/ou une PAD ≥ 90 (ou par un traitement antihypertenseur). Ayant uniquement la valeur de PAD dans cette échantillon, on sous-estime probablement la présence d'hypertension.

In [6]:
#Risque de diabète à 5 ans dans cet échantillon (268/768)
prévalence_diabète = (df["Outcome"].sum()) / (df["Outcome"].count())
print(f"Risque à 5 ans = {prévalence_diabète:.1%}")


Risque à 5 ans = 34.9%


Nous avons un risque à 5 ans de diabète qui est de 34,9 %. Selon la description Kaggle, Outcome indique un début de diabète dans les 5 ans ; la définition exacte étant incertaine, les 34,9 % sont à comparer avec prudence à une prévalence. On ne sait pas non plus si les patientes étaient diabétiques au départ.

In [7]:
#Comparer les moyennes et les médianes entre le groupe diabétique et le groupe non diabétique (version en une seule ligne)
comparaison_moy_2_grp = df.groupby("Outcome")[["Glucose", 'BloodPressure', "SkinThickness", "Insulin", "IMC", "DiabetesPedigreeFunction", "Age"]].agg(["mean", "median"])
print(comparaison_moy_2_grp)

            Glucose        BloodPressure        SkinThickness         \
               mean median          mean median          mean median   
Outcome                                                                
0        110.643863  107.0     70.877339   70.0     27.235457   27.0   
1        142.319549  140.0     75.321429   74.5     33.000000   32.0   

            Insulin               IMC        DiabetesPedigreeFunction         \
               mean median       mean median                     mean median   
Outcome                                                                        
0        130.287879  102.5  30.859674   30.1                 0.429734  0.336   
1        206.846154  169.5  35.406767   34.3                 0.550500  0.449   

               Age         
              mean median  
Outcome                    
0        31.190000   27.0  
1        37.067164   36.0  


On voit pour chaque valeur mesurée une moyenne et une médiane plus élevées dans le groupe diabétique. Concernant la DPF (Diabetes Pedigree Function), la moyenne pour des personnes non diabétiques est de 0,43 et pour les personnes diabétiques est de 0,55 ; nous avons donc des chiffres cohérents.


In [8]:
#Trouver les corrélations

test_de_correlation = df["Glucose"].corr(df.IMC)
print(test_de_correlation)

tableau_correlation = df.corr(numeric_only=True)["Outcome"].sort_values(ascending=False)
print(tableau_correlation)

0.23277051103551305
Outcome                     1.000000
Glucose                     0.494650
IMC                         0.313680
Insulin                     0.303454
SkinThickness               0.259491
Age                         0.238356
Pregnancies                 0.221898
DiabetesPedigreeFunction    0.173844
BloodPressure               0.170589
Name: Outcome, dtype: float64


Dans cet échantillon, la variable la plus associée au diabète est le glucose (r = 0,49), nettement devant l'IMC (0,31) et l'insuline (0,30). Cette association du glucose est attendue, puisque la glycémie fait partie des critères diagnostiques du diabète. Viennent ensuite l'épaisseur de peau, l'âge et le nombre de grossesses (0,22 à 0,26), puis la DPF et la pression artérielle (0,17). La corrélation de l'insuline repose sur environ la moitié des patientes seulement (394 patientes). Une corrélation mesure une association, pas une causalité.